# LLM · Lecture 11 — Agentic RAG & SQL-based RAG

Classic RAG **always** retrieves once, then answers. **Agentic RAG** lets the model *decide* whether, what, and when to retrieve — and choose *which* source: a **vector** search over prose, or **text-to-SQL** over structured tables. This notebook runs the whole lecture, with lots of examples, and ends with **scaffolded practice** you can complete in place.

1. classic vs agentic RAG — the model decides,
2. **retrieval as a tool** (the agent skips retrieval for chit-chat),
3. iterative retrieve → reflect → retrieve,
4–6. **text-to-SQL**: generate → guard → run → answer, with a reflect-and-retry on DB errors,
7. give the agent **both** tools and let it route,
8. a wider toolkit: schema-aware prompting, the SELECT-only guard vs hostile probes, parameterized queries, `JOIN`/`GROUP BY`/`HAVING`,
9. **practice** — 7 scaffolded exercises.

**Two kinds of cell.** The `sqlite3` parts (guards, injection, joins, most practice) are pure stdlib and **always run**. The parts that call the model (`gemma-4-E4B-it` via the class proxy) are **key-guarded** — paste your course API key to fire them. The retriever uses `bge-m3` embeddings when a key is set, and a small **offline fallback** otherwise, so the RAG structure runs either way.

## 0. Setup

In [ ]:
import os, re, json, math, sqlite3

os.environ.setdefault('OPENAI_BASE_URL', 'https://llm.nat-d.uk/v1')   # the class proxy
# os.environ['OPENAI_API_KEY'] = 'sk-...'   # <-- paste your key (portal: Profile -> API key)
# After pasting a key, RE-RUN THIS CELL so HAVE_KEY flips to True (the model + the
# bge-m3 retriever check it; the index below rebuilds itself when the mode changes).
MODEL = 'gemma-4-E4B-it'
HAVE_KEY = bool(os.environ.get('OPENAI_API_KEY'))

def client():
    from openai import OpenAI          # preinstalled in Colab
    return OpenAI()                    # reads OPENAI_API_KEY + OPENAI_BASE_URL

def need_key():
    print('No OPENAI_API_KEY set — set it in the cell above to run the live (model) cells.')
    print('(The sqlite cells below run regardless.)')

print('base:', os.environ['OPENAI_BASE_URL'], '| model:', MODEL, '| key set:', HAVE_KEY)

## 1. Classic RAG always retrieves — agentic RAG decides

Classic RAG is a fixed pipeline: **embed → search → stuff top-k into the prompt → answer**, every time. That wastes a retrieval on *"thanks!"*, and it can only look **once**. Agentic RAG wraps the retriever as a **tool** and drops it inside the agent loop from Lecture 10, so the model chooses whether to retrieve, with what query, from which source, and how many times. Same retriever — the *decision* is what's new.

## 2. Retrieval as a tool — the agent decides *when*

We give the model one tool, `search_docs`, via the native `tools=` path. For chit-chat it emits **no** tool call and just replies; for a factual question it calls `search_docs`, we run the retriever, feed the passage back as a `role:"tool"` message, and it answers **grounded** in that passage. First the retriever (works offline via the fallback):

In [ ]:
KB = [   # (source tag, passage) — we return the source so the answer can CITE it (lecture §2)
    ("refund-policy",   "Refunds are issued within 14 days of purchase to the original card."),
    ("shipping-policy", "Standard shipping takes 3-5 business days; express is next-day."),
    ("warranty-policy", "All products carry a 1-year limited warranty against defects."),
]
DOCS = [text for _, text in KB]

def _ngrams(s, n=3):
    s = re.sub(r"\s+", " ", " " + s.lower() + " ")
    return [s[i:i+n] for i in range(len(s) - n + 1)]

def _local_embed(texts, dim=512):        # offline fallback: char-trigram bag-of-words
    out = []
    for t in texts:
        v = [0.0]*dim
        for g in _ngrams(t):
            v[hash(g) % dim] += 1.0
        out.append(v)
    return out

def embed(texts):
    if HAVE_KEY:
        return [d.embedding for d in client().embeddings.create(model='bge-m3', input=texts).data]
    return _local_embed(texts)           # no key -> local fallback so search still runs

def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na = math.sqrt(sum(x*x for x in a)); nb = math.sqrt(sum(y*y for y in b))
    return dot/(na*nb) if na and nb else 0.0

# The index MUST be embedded the same way as the query, else cosine() compares
# vectors of different lengths and ranking becomes garbage with NO error. So we tag
# the index with the mode it was built in and rebuild it when HAVE_KEY changes
# (e.g. you ran offline first, then pasted a key and re-ran the setup cell).
_INDEX = {"mode": None, "vecs": None}

def index_vecs():
    if _INDEX["mode"] != HAVE_KEY:        # first call, or key added since -> (re)build
        _INDEX["vecs"], _INDEX["mode"] = embed(DOCS), HAVE_KEY
    return _INDEX["vecs"]

def search_docs(query, k=1):
    qv = embed([query])[0]
    ranked = sorted(((cosine(qv, v), src, text) for v, (src, text) in zip(index_vecs(), KB)),
                    reverse=True)
    return " ".join(f"{text} [source: {src}]" for _, src, text in ranked[:k]) or "no documents"

print(search_docs("refund policy"))
print(search_docs("how fast is shipping"))

Now the **agentic decision**: one tool, the native `tools=` path. `ask()` runs a bounded loop and reports whether the model chose to retrieve.

In [ ]:
tool_specs = [
    {"type": "function", "function": {
        "name": "search_docs",
        "description": ("Search the knowledge base for facts about policies "
                        "(refunds, shipping, warranty). Call ONLY when you need "
                        "facts you don't already have."),
        "parameters": {"type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"]}}},
]

def ask(user_msg, max_steps=3):
    c = client()
    messages = [{"role": "user", "content": user_msg}]
    retrieved = False
    for _ in range(max_steps):                       # bounded -> always terminates
        msg = c.chat.completions.create(
            model=MODEL, messages=messages, tools=tool_specs, temperature=0).choices[0].message
        if not getattr(msg, "tool_calls", None):
            return ("retrieved" if retrieved else "NO retrieval", msg.content or "")
        retrieved = True
        messages.append(msg)                         # the assistant's tool request
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = search_docs(args.get("query", ""))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return ("hit max_steps", "")

if HAVE_KEY:
    print(ask("Thanks, that's all!"))         # -> ('NO retrieval', ...)
    print(ask("What is your refund policy?"))  # -> ('retrieved', 'Refunds ...')
else:
    need_key()

Same code, opposite behaviour, decided by the model: chit-chat → no retrieval; a factual question → `search_docs` then a grounded answer. **That is agentic RAG in one function.**

## 3. Iterative retrieval: retrieve → reflect → retrieve again

Because retrieval is a tool inside a loop, the agent can look **more than once**: retrieve, read the passages, then retrieve again with a refined query before answering — e.g. *"compare the sampling settings in Lecture 1 and Lecture 8"* needs two searches. Guard it: **cap the iterations** (the `max_steps` above), **detect repeated queries**, and **log every query + result** as your debugging trace.

## 4–5. Text-to-SQL: retrieving from structured data

Counts, sums, filters and joins live in **tables**, and embedding rows is the wrong tool. Instead the model **writes a `SELECT`**, we run it, and feed the rows back:

`"Total units in the North?"` → `SELECT sum(units) FROM sales WHERE region='North'` → `[(200,)]` → *"The North sold 200 units."*

The big win: **the database does the arithmetic exactly** (LLMs are unreliable at summing). Here is the full loop — a tiny in-memory DB, the schema in the prompt, a **guard**, `LIMIT` injection, and a **reflect-and-retry** if SQLite raises an error.

In [ ]:
# 1. a tiny in-memory "database"
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE sales(region TEXT, product TEXT, units INT)")
db.executemany("INSERT INTO sales VALUES (?,?,?)", [
    ("North", "Widget", 120), ("North", "Gadget", 80),
    ("South", "Widget", 45),  ("South", "Gadget", 200),
])
db.commit()
SCHEMA = "CREATE TABLE sales(region TEXT, product TEXT, units INT)"

# keywords that must never appear, even after a leading SELECT (defense in depth)
BANNED = re.compile(r"(?is)\b(insert|update|delete|drop|alter|create|attach|"
                    r"detach|pragma|replace|vacuum|reindex)\b")

def clean(text):                          # strip ``` fences / a leading 'sql' / trailing ;
    sql = (text or "").strip().strip("`")
    return re.sub(r"^sql\b", "", sql, flags=re.I).strip().rstrip(";").strip()

def guard(sql):                           # returns a reason string if unsafe, else None
    if not re.match(r"(?is)^\s*select\b", sql):
        return "not a SELECT"
    if ";" in sql:
        return "multiple statements"
    if BANNED.search(sql):
        return "banned keyword"
    return None

def add_limit(sql, n=100):                # cap rows if the query has no LIMIT
    return sql if re.search(r"(?is)\blimit\b", sql) else sql + f" LIMIT {n}"

def run_guarded(sql):
    bad = guard(sql)
    if bad:
        raise ValueError(f"Refused ({bad}): {sql}")
    return db.execute(add_limit(sql)).fetchall()

def gen_sql(question, error=None):        # model writes ONE SELECT from the schema
    msgs = [
        {"role": "system", "content":
            "You write a single SQLite SELECT query for the user's question. "
            "Output ONLY the SQL, no markdown, no explanation. Schema:\n" + SCHEMA},
        {"role": "user", "content": question},
    ]
    if error:                             # reflect-and-retry: feed the DB error back
        msgs.append({"role": "user", "content": f"That query failed with: {error}\nFix it. SQL only."})
    return clean(client().chat.completions.create(model=MODEL, temperature=0, messages=msgs)
                 .choices[0].message.content)

def text_to_sql(question):
    sql = gen_sql(question)
    print("Generated SQL:", sql)
    try:
        rows = run_guarded(sql)
    except sqlite3.Error as e:            # bad column / typo -> one retry with the error fed back
        print("DB error:", e, "-> retrying")
        rows = run_guarded(gen_sql(question, error=str(e)))
    print("Rows:", rows)
    answer = client().chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "Answer in one sentence using ONLY the rows."},
        {"role": "user", "content": f"Q: {question}\nRows: {rows}"}]).choices[0].message.content or ""
    print("Answer:", answer)
    return rows

if HAVE_KEY:
    text_to_sql("Which region sold the most units in total?")   # -> South
else:
    need_key()
    print("\n(No key: the SQL/guard/JOIN cells below still run — the model only writes the query.)")

**Temperature 0** for the SQL step (deterministic, correct query); a separate call renders the rows as prose. The model never aggregates — `sqlite3` does. The `try/except` is the **reflect-and-retry**: a hallucinated column raises `no such column`, we hand that message back, and the model corrects itself.

## 6. Guarding text-to-SQL — generated SQL is *untrusted input*

Layer **independent** defences: a **read-only connection** (the real protection in production), **single statement** (`sqlite3` also refuses two), a **keyword denylist**, **schema in the prompt**, and a forced **`LIMIT`**. And never build SQL from user text yourself — the classic injection, side by side (pure stdlib, always runs):

In [ ]:
d = sqlite3.connect(":memory:")
d.execute("CREATE TABLE sales(region TEXT, units INT)")
d.executemany("INSERT INTO sales VALUES (?,?)", [("North", 120), ("South", 45)])

user_region = "North' OR '1'='1"                       # hostile input

bad = f"SELECT * FROM sales WHERE region = '{user_region}'"     # WRONG: f-string glue
print("f-string (WRONG):", d.execute(bad).fetchall())          # -> leaks BOTH rows

good = "SELECT * FROM sales WHERE region = ?"                    # RIGHT: bind parameter
print("parameter (RIGHT):", d.execute(good, (user_region,)).fetchall())   # -> [] (matched as a value)

The difference is **who builds the string**. In text-to-SQL the *model* writes the `SELECT` and your code **gates** it; the moment *you* f-string user text into a query, you've reintroduced the exact bug parameters exist to kill.

**The regex is a heuristic, not the security boundary.** `^\s*select` also **false-rejects** a perfectly safe read-only CTE — `WITH t AS (SELECT ...) SELECT * FROM t` starts with `WITH`, so the guard refuses a legitimate query. The layer that *actually* makes writes impossible is a **read-only database role/connection**; the regex + denylist are cheap defense-in-depth on top. Widen the guard to allow a leading `WITH` when you need CTEs — but keep the read-only role.

## 7. Vector search or SQL? Give the agent *both*

They're complementary retrievers. Expose **both** as tools and let the model route:

| | Best for | Example |
|---|---|---|
| **Vector search** | prose: notes, PDFs, docs | "what does the lecture say about nucleus sampling?" |
| **Text-to-SQL** | tables: counts, sums, joins | "how many orders shipped last week?" |

The model picks `search_docs` for a conceptual question and a SQL tool for a quantitative one — and calls each in turn for a question needing both. That routing is just the Lecture-10 agent loop choosing among tools (you build it in Practice 5).

## 8. A wider toolkit (mostly pure `sqlite3` — runs with no key)

### 8.1 Schema-aware prompting — hand the model the `CREATE TABLE`, not a sentence
A vague description makes the model **guess** column names (a runtime error, or silently wrong rows). Paste the literal DDL — SQLite hands it to you verbatim from `sqlite_master`:

In [ ]:
db2 = sqlite3.connect(":memory:")
db2.executescript("""
CREATE TABLE customers(id INTEGER PRIMARY KEY, name TEXT, region TEXT);
CREATE TABLE orders(id INTEGER PRIMARY KEY, customer_id INTEGER, units INT,
                    FOREIGN KEY(customer_id) REFERENCES customers(id));
""")
schema_prompt = "\n".join(r[0] for r in db2.execute(
    "SELECT sql FROM sqlite_master WHERE type='table'"))
print(schema_prompt)     # <- THIS is your schema prompt: exact names, types, foreign keys

### 8.2 The SELECT-only guard vs hostile probes
See exactly *which layer* refuses each attack (pure stdlib):

In [ ]:
def which_layer(sql):
    if not re.match(r"(?is)^\s*select\b", sql): return "layer 1: not a SELECT"
    if ";" in sql:                                return "layer 2: multiple statements"
    if BANNED.search(sql):                        return "layer 3: banned keyword"
    return "PASSED (a plain read) — rely on the read-only role + table allow-list"

probes = [
    "SELECT region, SUM(units) FROM sales GROUP BY region",   # legit read -> PASSES all layers
    "DROP TABLE customers",
    "PRAGMA table_info(orders)",
    "ATTACH DATABASE '/etc/passwd' AS x",
    "SELECT 1; DROP TABLE customers",
    "SELECT * FROM orders; PRAGMA writable_schema=ON",
    "select * from orders -- DROP TABLE x",
]
for p in probes:
    print(f"{which_layer(p):48}  <-  {p}")

### 8.3 GROUP BY + JOIN + HAVING — what the database does that the model shouldn't
These are the ground truth text-to-SQL is *trying to generate* — every number computed exactly by SQLite (pure stdlib):

In [ ]:
db2.executemany("INSERT INTO customers VALUES (?,?,?)",
                [(1, "Acme", "North"), (2, "Globex", "South"), (3, "Initech", "North")])
db2.executemany("INSERT INTO orders VALUES (?,?,?)",
                [(1, 1, 120), (2, 1, 30), (3, 2, 200), (4, 3, 45)])

print("units per region:", db2.execute("""
    SELECT c.region, SUM(o.units) AS total
    FROM orders o JOIN customers c ON c.id = o.customer_id
    GROUP BY c.region ORDER BY total DESC
""").fetchall())                          # -> [('South', 200), ('North', 195)]

print("customers over 100 units:", db2.execute("""
    SELECT c.name, SUM(o.units) AS total
    FROM customers c JOIN orders o ON o.customer_id = c.id
    GROUP BY c.id HAVING total > 100 ORDER BY total DESC
""").fetchall())                          # -> [('Globex', 200), ('Acme', 150)]

`JOIN` stitches the tables on the foreign key; `GROUP BY` buckets; `SUM` adds; **`HAVING` filters the *aggregated* groups** (`WHERE` can't — it runs before aggregation). Model translates, database computes.

### 8.4 The when-to-retrieve gate, in isolation (optional model call)
A one-token router — `RETRIEVE` or `ANSWER` — the cheapest "do I need facts?" gate, easy to log and evaluate:

In [ ]:
def route(question):
    content = client().chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content":
            "Reply with exactly one word: RETRIEVE if answering needs facts from the "
            "knowledge base, or ANSWER if it's greeting/chit-chat/something you can "
            "answer directly. One word only."},
        {"role": "user", "content": question}]).choices[0].message.content or ""
    return content.strip().upper()

if HAVE_KEY:
    for q in ["Hi there!", "Thanks, that's all!",
              "What is your refund policy?", "How many orders shipped last week?"]:
        print(f"{route(q):>9}  <-  {q}")
else:
    need_key()

## 9. Practice — scaffolded

Each exercise below has the setup ready; fill in the `TODO` and run. Most are pure `sqlite3` and work with **no key**; three touch the model — **Practice 1** (a hybrid: its target SQL runs offline, the model call is optional) and **Practice 4 & 5** (need a key, marked).

**Practice 1 — a `GROUP BY` question through text-to-SQL.** Change the question to one that needs a `GROUP BY` and confirm the generated query passes the guard and returns correct rows. (Needs a key for the model; the target SQL is also run directly so you see the answer regardless.)

In [ ]:
# TODO: set a GROUP BY question and (with a key) run it through the pipeline
question = "How many units of each product?"

target = "SELECT product, SUM(units) FROM sales GROUP BY product ORDER BY 2 DESC"
print("guard(target) ->", guard(target), "| rows:", db.execute(add_limit(target)).fetchall())
assert guard(target) is None and db.execute(target).fetchall(), "target should pass + return rows"
if HAVE_KEY:
    text_to_sql(question)     # the model should generate an equivalent GROUP BY query

**Practice 2 — block `DROP` / `PRAGMA`.** Confirm the guard refuses a destructive query and a smuggled `PRAGMA` after a leading `SELECT`. (Pure stdlib.)

In [ ]:
# TODO: add one more malicious string you think should be refused, then run
attacks = [
    "DROP TABLE sales",                      # refused at layer 1 (not a SELECT)
    "SELECT 1; PRAGMA writable_schema=ON",   # refused at layer 2 (;) — short-circuits before the denylist
    # "<your attack here>",
]
for a in attacks:
    print(f"{str(guard(a)):22}  <-  {a}")
    assert guard(a) is not None, f"guard must refuse: {a}"
print("all refused ✓")

**Practice 3 — require a `LIMIT`.** `add_limit` appends one only when absent (and after `clean()` stripped any trailing `;`, so you never get `...; LIMIT 100`). (Pure stdlib.)

In [ ]:
print(add_limit("SELECT * FROM sales"))                 # -> ... LIMIT 100
print(add_limit("SELECT * FROM sales LIMIT 5"))         # -> unchanged
print(add_limit(clean("SELECT * FROM sales;")))         # -> clean() drops ';' first, then LIMIT
assert add_limit("SELECT 1").endswith("LIMIT 100")     # none present -> appended
assert add_limit("select 1 limit 3") == "select 1 limit 3"   # already capped -> unchanged
print("LIMIT logic ✓")

**Practice 4 — agentic retrieval (the HW6 idea).** Reuse the §2 `ask()` agent: chit-chat gets **no** retrieval (fast), a factual question triggers `search_docs`. Compare. (Needs a key.)

In [ ]:
if HAVE_KEY:
    for q in ["Hey, thanks!", "What is your warranty?"]:
        tag, ans = ask(q)
        print(f"[{tag:12}] {q!r} -> {ans[:70]!r}")
else:
    need_key()
    print("Offline, you can still call search_docs directly:", search_docs("warranty")[:50])

**Practice 5 — route across TWO tools.** Give the agent both `search_docs` and a SQL runner and confirm it routes a conceptual question to one and a counting question to the other. A `run_sql` tool + dispatch are scaffolded; complete the `tools` list. (Needs a key.)

In [ ]:
def run_sql(query):                       # a guarded SQL tool over the `sales` table
    bad = guard(clean(query))
    if bad: return f"refused: {bad}"
    try:   return str(db.execute(add_limit(clean(query))).fetchall())
    except sqlite3.Error as e: return f"error: {e}"

TWO_TOOLS = [
    tool_specs[0],                        # search_docs (from §2)
    # TODO: add the run_sql tool spec (name 'run_sql', one string arg 'query',
    #       description: "Run a read-only SQL SELECT over sales(region, product, units)").
]

def two_tool_agent(question, max_steps=4):
    c = client(); messages = [{"role": "user", "content": question}]
    dispatch = {"search_docs": lambda a: search_docs(a.get("query", "")),
                "run_sql": lambda a: run_sql(a.get("query", ""))}
    for _ in range(max_steps):
        msg = c.chat.completions.create(model=MODEL, messages=messages,
                                        tools=TWO_TOOLS, temperature=0).choices[0].message
        if not getattr(msg, "tool_calls", None): return msg.content or ""
        messages.append(msg)
        for call in msg.tool_calls:
            a = json.loads(call.function.arguments or "{}")
            print(f"  -> {call.function.name}({a})")
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": dispatch[call.function.name](a)})
    return "hit max_steps"

if HAVE_KEY and len(TWO_TOOLS) == 2:
    print(two_tool_agent("What is our refund policy?"))       # -> search_docs
    print(two_tool_agent("How many total units were sold?"))  # -> run_sql
else:
    print("Add the run_sql tool spec (and set a key) to run the router.")

**Practice 6 — which layer refuses each probe.** Run `which_layer` over the probes, then add an **8th** probe that *passes* layer 1 (starts with `SELECT`) but should still be refused — a cross-table `UNION` exfiltration — and decide how you'd stop it. (Pure stdlib.)

In [ ]:
# TODO: add a UNION-based probe that starts with SELECT and has no ';'
eighth = "SELECT * FROM orders WHERE 1=1 UNION SELECT name FROM customers"   # passes layers 1-3!
print(f"{which_layer(eighth):48}  <-  {eighth}")
print("It PASSES the regex guard — a plain read. Stop cross-table exfiltration with the")
print("read-only role + an ALLOW-LIST of exposed tables (and/or add 'union' to the denylist).")

**Practice 7 — `JOIN`+`WHERE` vs `JOIN`+`HAVING`.** Fill in two queries on the two-table DB: the North total (filter *rows* → `WHERE`), and regions over 150 units (filter the *aggregate* → `HAVING`). (Pure stdlib.)

In [ ]:
north_total = """
    -- TODO: SUM units for customers in the North (JOIN + WHERE)
    SELECT SUM(o.units)
    FROM orders o JOIN customers c ON c.id = o.customer_id
    WHERE c.region = 'North'
"""
regions_over_150 = """
    -- TODO: regions whose customers ordered > 150 units in total (JOIN + GROUP BY + HAVING)
    SELECT c.region, SUM(o.units) AS total
    FROM orders o JOIN customers c ON c.id = o.customer_id
    GROUP BY c.region HAVING total > 150 ORDER BY total DESC
"""
print("North total:", db2.execute(north_total).fetchone())          # -> (195,)
print("regions > 150:", db2.execute(regions_over_150).fetchall())   # -> [('South', 200), ('North', 195)]
print("Why HAVING? The per-region SUM doesn't exist until AFTER GROUP BY, so WHERE can't see it.")

## Recap
- **Agentic RAG** = the model *decides* whether/what/when to retrieve (retrieval-as-a-tool inside the agent loop), and can retrieve → reflect → retrieve again (bounded + logged).
- **Text-to-SQL** = model writes a `SELECT`, your code **guards + runs** it, rows fed back for a grounded answer. **The database does the arithmetic**, not the model.
- **Guard generated SQL** like untrusted input: read-only role, single statement, keyword denylist, schema-in-prompt, forced `LIMIT`; **never f-string user text** — bind with `?`.
- Give the agent **both** a vector retriever and a SQL runner and let it **route**. `WHERE` filters rows; **`HAVING` filters aggregates**.